# Gradient Descent vs Normal Equation — Interactive Walkthrough

This notebook is a guided tour of the same experiments that live in `experiments/`. It is meant to be run cell-by-cell so you can poke at the parameters and watch the geometry / convergence change in real time.

Sections:
1. Setup
2. Correctness sanity check
3. Loss surface geometry — scaled vs unscaled
4. GD trajectories on top of the loss surface
5. Iterations to converge as feature-scale spread grows
6. Wall-clock benchmark (mini-version)
7. The conditioning failure case
8. Try-it-yourself ideas

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from src.data_gen import make_regression, make_collinear, standardize
from src.normal_equation import fit_normal_equation, fit_pinv
from src.gradient_descent import fit_gradient_descent, safe_lr
from src.utils import setup_plot_style, condition_number

setup_plot_style()

## 2. Correctness sanity check

Both methods should land on the same `theta` to ~machine precision when the data is well-conditioned.

In [ ]:
X, y, theta_true = make_regression(n=2000, p=20, noise=0.1, seed=42)
X, _, _ = standardize(X)

theta_ne = fit_normal_equation(X, y)
res = fit_gradient_descent(X, y, lr=safe_lr(X) * 0.9, max_iters=50_000, tol=1e-12)

print(f'GD iterations: {res.n_iters}')
print(f'||theta_GD - theta_NE|| = {np.linalg.norm(res.theta - theta_ne):.2e}')
print(f'||theta_NE - theta_true|| (residual noise) = {np.linalg.norm(theta_ne - theta_true):.4f}')

## 3. Loss surface geometry

We evaluate the loss on a 2D grid in parameter space and contour-plot it. The unscaled version produces a long thin valley; standardization turns it into a near-circular bowl.

In [ ]:
rng = np.random.default_rng(0)
n = 400
x1 = rng.standard_normal(n)
x2 = rng.standard_normal(n) * 50.0   # try changing this scale!
X_u = np.column_stack([x1, x2])
theta_true = np.array([2.0, -1.0])
y = X_u @ theta_true + 0.05 * rng.standard_normal(n)
X_s, _, _ = standardize(X_u)

print(f'kappa unscaled    = {condition_number(X_u):.2e}')
print(f'kappa standardized = {condition_number(X_s):.2e}')

def loss_grid(X, y, center, span=2.5, n=120):
    t1 = np.linspace(center[0]-span, center[0]+span, n)
    t2 = np.linspace(center[1]-span, center[1]+span, n)
    T1, T2 = np.meshgrid(t1, t2)
    L = np.zeros_like(T1)
    for i in range(n):
        for j in range(n):
            r = X @ np.array([T1[i,j], T2[i,j]]) - y
            L[i,j] = 0.5 * np.mean(r**2)
    return T1, T2, L

fig, axes = plt.subplots(1, 2, figsize=(13,5))
for ax, X, title in [(axes[0], X_u, 'Unscaled'), (axes[1], X_s, 'Standardized')]:
    ts = fit_normal_equation(X, y)
    T1, T2, L = loss_grid(X, y, ts)
    ax.contour(T1, T2, np.log10(L + 1e-12), levels=20, cmap='viridis')
    ax.plot(*ts, 'r*', markersize=14)
    ax.set_title(title); ax.set_aspect('equal', adjustable='datalim')
plt.show()

## 4. GD trajectories

Now overlay an actual GD run on each contour plot. On the unscaled data, GD zig-zags across the narrow valley; on standardized data, it walks straight to the minimum.

In [ ]:
init_offset = np.array([2.0, 2.0])
fig, axes = plt.subplots(1, 2, figsize=(13,5))
for ax, X, title in [(axes[0], X_u, 'Unscaled (zig-zag)'), (axes[1], X_s, 'Scaled (straight)')]:
    ts = fit_normal_equation(X, y)
    res = fit_gradient_descent(X, y, lr=safe_lr(X)*0.9, max_iters=200,
                                theta_init=ts+init_offset,
                                record_trajectory=True, tol=0)
    T1, T2, L = loss_grid(X, y, ts, span=3.0)
    ax.contour(T1, T2, np.log10(L + 1e-12), levels=20, cmap='viridis', alpha=0.6)
    traj = np.array(res.theta_history)
    ax.plot(traj[:,0], traj[:,1], 'o-', color='crimson', markersize=3, linewidth=1)
    ax.plot(*ts, 'k*', markersize=14)
    ax.set_title(title); ax.set_aspect('equal', adjustable='datalim')
plt.show()

## 5. The headline result — iterations vs feature-scale spread

This is the centerpiece. As the ratio of largest to smallest feature scale grows, GD on raw data needs orders of magnitude more iterations. Standardized GD barely moves.

In [ ]:
spreads = np.logspace(0, 5, 11)
iters_unscaled, iters_scaled = [], []
n, p = 1000, 10

for spread in spreads:
    scales = np.logspace(0, np.log10(spread), p)
    X, y, _ = make_regression(n=n, p=p, seed=0, feature_scales=scales)
    X_s, _, _ = standardize(X)
    res_u = fit_gradient_descent(X, y, lr=safe_lr(X)*0.9, max_iters=200_000, tol=1e-6)
    res_s = fit_gradient_descent(X_s, y, lr=safe_lr(X_s)*0.9, max_iters=200_000, tol=1e-6)
    iters_unscaled.append(res_u.n_iters)
    iters_scaled.append(res_s.n_iters)

plt.loglog(spreads, iters_unscaled, marker='o', label='unscaled')
plt.loglog(spreads, iters_scaled, marker='s', label='standardized')
plt.xlabel('max/min feature scale ratio'); plt.ylabel('iterations to converge')
plt.legend(); plt.title('Scaling collapses convergence cost')
plt.show()

## 6. Mini benchmark — wall time vs p

In [ ]:
import time

p_grid = [50, 100, 250, 500, 1000, 2000]
t_ne, t_gd = [], []
for p in p_grid:
    X, y, _ = make_regression(n=5000, p=p, seed=0)
    X, _, _ = standardize(X)
    lr = safe_lr(X) * 0.9
    t0 = time.perf_counter(); fit_normal_equation(X, y); t_ne.append(time.perf_counter()-t0)
    t0 = time.perf_counter(); fit_gradient_descent(X, y, lr=lr, max_iters=500); t_gd.append(time.perf_counter()-t0)

plt.loglog(p_grid, np.array(t_ne)*1e3, marker='o', label='Normal Equation')
plt.loglog(p_grid, np.array(t_gd)*1e3, marker='s', label='GD (500 iters)')
plt.xlabel('p'); plt.ylabel('wall time (ms)'); plt.legend(); plt.title('Wall time vs p')
plt.show()

## 7. Conditioning — where NE quietly returns wrong answers

In [ ]:
kappa_grid = np.logspace(1, 9, 9)
err_ne, err_pinv, err_gd = [], [], []
for kappa in kappa_grid:
    X, y, theta_true = make_regression(n=1000, p=30, condition_number=kappa, seed=0)
    err_ne.append(np.linalg.norm(fit_normal_equation(X, y) - theta_true))
    err_pinv.append(np.linalg.norm(fit_pinv(X, y) - theta_true))
    res = fit_gradient_descent(X, y, lr=safe_lr(X)*0.5, max_iters=20_000, tol=1e-10)
    err_gd.append(np.linalg.norm(res.theta - theta_true))

plt.loglog(kappa_grid, err_ne, marker='o', label='Normal Equation')
plt.loglog(kappa_grid, err_pinv, marker='s', label='pinv')
plt.loglog(kappa_grid, err_gd, marker='^', label='GD (early-stopped)')
plt.xlabel(r'$\kappa(X^\top X)$'); plt.ylabel(r'$\|\hat\theta - \theta^*\|$')
plt.legend(); plt.title('Recovery error vs conditioning')
plt.show()

## 8. Try it yourself

- In Section 3, change `x2 = rng.standard_normal(n) * 50.0` to `* 1.0` and re-run. The contours go circular and the trajectory plot in Section 4 walks straight even on "unscaled" data.
- In Section 4, push the learning rate above `safe_lr(X)` (try `lr=safe_lr(X) * 1.05`). GD diverges — you'll see the trajectory fly off the contour plot.
- In Section 5, increase `p` from 10 to 100. The unscaled curve gets even worse.
- In Section 7, swap `fit_pinv(X, y)` for `fit_pinv(X, y, rcond=1e-6)` and watch the SVD-based solver suddenly track GD by truncating tiny singular values.